# Experiment 8: Clustering Human Activity Recognition Data (K-Means, DBSCAN, and Hierarchical Clustering)
This standalone notebook implements unsupervised clustering models (K-Means with Elbow & Silhouette optimization, DBSCAN with noise filtering, and Hierarchical Agglomerative Clustering with Ward's linkage) on the 561-feature UCI HAR sensory dataset. All figures strictly adhere to Sharruk's ML Lab Guidelines (Times New Roman, 15 pt bold labels, 600 DPI vector EPS).

In [1]:
import os
import warnings
import matplotlib
matplotlib.use('Agg') # Headless non-interactive backend
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path
from scipy.cluster.hierarchy import dendrogram, linkage
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, DBSCAN, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.metrics import (
    silhouette_score, davies_bouldin_score, calinski_harabasz_score,
    adjusted_rand_score, normalized_mutual_info_score
)

# Enforce Sharruk's Guidelines: Times New Roman, 15 pt base, Bold 15 pt X/Y labels, 15 pt legends
plt.rcParams.update({
    'font.family': 'Times New Roman',
    'font.size': 15,
    'axes.labelsize': 15,
    'axes.labelweight': 'bold',
    'axes.titlesize': 15,
    'axes.titleweight': 'bold',
    'xtick.labelsize': 13,
    'ytick.labelsize': 13,
    'legend.fontsize': 15,
    'figure.titlesize': 16,
    'figure.titleweight': 'bold'
})
warnings.filterwarnings('ignore')

def resolve_path(rel_path):
    for prefix in ['', '../', '../../']:
        cand = os.path.join(prefix, rel_path)
        if os.path.exists(cand):
            return cand
    return rel_path

def resolve_out(rel_path):
    if os.path.basename(os.getcwd()) == 'Ex8':
        if rel_path.startswith('Ex8/'):
            return rel_path[len('Ex8/'):]
    return rel_path

os.makedirs(resolve_out('Ex8'), exist_ok=True)

In [2]:
def run_experiment_8(data_dir='Datasets/UCI_HAR', sample_size=3000):
    print('='*60)
    print('=== LAUNCHING EXPERIMENT 8: HAR CLUSTERING PIPELINE ===')
    print('='*60)
    dir_path = Path(resolve_path(data_dir))
    X_train = pd.read_csv(dir_path / 'train/X_train.txt', delim_whitespace=True, header=None).values
    y_train = pd.read_csv(dir_path / 'train/y_train.txt', delim_whitespace=True, header=None).values.ravel()
    X_test = pd.read_csv(dir_path / 'test/X_test.txt', delim_whitespace=True, header=None).values
    y_test = pd.read_csv(dir_path / 'test/y_test.txt', delim_whitespace=True, header=None).values.ravel()

    X_full = np.vstack((X_train, X_test))
    y_full = np.concatenate((y_train, y_test))

    np.random.seed(42)
    sample_idx = np.random.choice(len(X_full), size=sample_size, replace=False)
    X_sample = X_full[sample_idx]
    y_sample = y_full[sample_idx]

    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(X_sample)

    # 1. Elbow & Silhouette Curves
    ks = range(2, 9)
    inertias, silhouettes = [], []
    for k in ks:
        km = KMeans(n_clusters=k, random_state=42, n_init=10)
        labels = km.fit_predict(X_scaled)
        inertias.append(km.inertia_)
        silhouettes.append(silhouette_score(X_scaled, labels))

    # Save Elbow Curve (600 DPI EPS)
    plt.figure(figsize=(9, 6))
    plt.plot(ks, inertias, marker='o', linewidth=2.5, markersize=8, color='#2b5c8f')
    plt.title('K-Means Elbow Curve (Inertia vs. Clusters)', fontsize=16, fontweight='bold')
    plt.xlabel('Number of Clusters (k)', fontsize=15, fontweight='bold')
    plt.ylabel('Inertia (WCSS)', fontsize=15, fontweight='bold')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(resolve_out('Ex8/Elbow_Curve.eps'), format='eps', dpi=600)
    plt.close()

    # Save Silhouette Curve (600 DPI EPS)
    plt.figure(figsize=(9, 6))
    plt.plot(ks, silhouettes, marker='s', linewidth=2.5, markersize=8, color='#d95f02')
    plt.title('K-Means Silhouette Score vs. Clusters', fontsize=16, fontweight='bold')
    plt.xlabel('Number of Clusters (k)', fontsize=15, fontweight='bold')
    plt.ylabel('Silhouette Coefficient', fontsize=15, fontweight='bold')
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.savefig(resolve_out('Ex8/Silhouette_Curve.eps'), format='eps', dpi=600)
    plt.close()

    # 2. Dendrogram (Ward Linkage)
    subset_dendro = X_scaled[:150]
    linkage_mat = linkage(subset_dendro, method='ward')
    plt.figure(figsize=(12, 6))
    dendrogram(linkage_mat, truncate_mode='lastp', p=30, leaf_rotation=90, leaf_font_size=11)
    plt.title("Hierarchical Agglomerative Dendrogram (Ward's Linkage)", fontsize=16, fontweight='bold')
    plt.xlabel('Cluster Size / Sample Index', fontsize=15, fontweight='bold')
    plt.ylabel('Ward Distance', fontsize=15, fontweight='bold')
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(resolve_out('Ex8/Dendrogram.eps'), format='eps', dpi=600)
    plt.close()

    # 3. 2D Cluster Scatter Comparison
    pca = PCA(n_components=2, random_state=42)
    X_pca2 = pca.fit_transform(X_scaled)
    km_6 = KMeans(n_clusters=6, random_state=42, n_init=10).fit_predict(X_scaled)
    db = DBSCAN(eps=15.0, min_samples=15).fit_predict(X_scaled)
    hac_ward = AgglomerativeClustering(n_clusters=6, linkage='ward').fit_predict(X_scaled)

    fig, axes = plt.subplots(2, 2, figsize=(16, 12))
    axes[0, 0].scatter(X_pca2[:, 0], X_pca2[:, 1], c=y_sample, cmap='tab10', s=20, alpha=0.65)
    axes[0, 0].set_title('Ground Truth Activities (6 Classes)', fontsize=15, fontweight='bold')
    axes[0, 0].set_xlabel('PC 1', fontsize=15, fontweight='bold')
    axes[0, 0].set_ylabel('PC 2', fontsize=15, fontweight='bold')
    axes[0, 0].grid(True, alpha=0.3)

    axes[0, 1].scatter(X_pca2[:, 0], X_pca2[:, 1], c=km_6, cmap='tab10', s=20, alpha=0.65)
    axes[0, 1].set_title('K-Means Clusters (k=6)', fontsize=15, fontweight='bold')
    axes[0, 1].set_xlabel('PC 1', fontsize=15, fontweight='bold')
    axes[0, 1].set_ylabel('PC 2', fontsize=15, fontweight='bold')
    axes[0, 1].grid(True, alpha=0.3)

    axes[1, 0].scatter(X_pca2[:, 0], X_pca2[:, 1], c=db, cmap='tab10', s=20, alpha=0.65)
    axes[1, 0].set_title('DBSCAN Clusters (eps=15, minPts=15)', fontsize=15, fontweight='bold')
    axes[1, 0].set_xlabel('PC 1', fontsize=15, fontweight='bold')
    axes[1, 0].set_ylabel('PC 2', fontsize=15, fontweight='bold')
    axes[1, 0].grid(True, alpha=0.3)

    axes[1, 1].scatter(X_pca2[:, 0], X_pca2[:, 1], c=hac_ward, cmap='tab10', s=20, alpha=0.65)
    axes[1, 1].set_title('Hierarchical Clustering (Ward, k=6)', fontsize=15, fontweight='bold')
    axes[1, 1].set_xlabel('PC 1', fontsize=15, fontweight='bold')
    axes[1, 1].set_ylabel('PC 2', fontsize=15, fontweight='bold')
    axes[1, 1].grid(True, alpha=0.3)

    plt.suptitle('2D PCA Projections: Ground Truth vs. Clustering Models', fontsize=17, fontweight='bold')
    plt.tight_layout()
    plt.savefig(resolve_out('Ex8/Cluster_Scatter.eps'), format='eps', dpi=600)
    plt.close()

    # 4. Clustering Comparison Bar Chart
    models_cl = ['K-Means (k=6)', 'DBSCAN', 'Hierarchical (Ward)']
    ari_scores = [adjusted_rand_score(y_sample, km_6), adjusted_rand_score(y_sample, db), adjusted_rand_score(y_sample, hac_ward)]
    nmi_scores = [normalized_mutual_info_score(y_sample, km_6), normalized_mutual_info_score(y_sample, db), normalized_mutual_info_score(y_sample, hac_ward)]
    sil_scores = [silhouette_score(X_scaled, km_6), silhouette_score(X_scaled, db), silhouette_score(X_scaled, hac_ward)]

    x_cl = np.arange(len(models_cl))
    w_cl = 0.25
    plt.figure(figsize=(11, 6))
    plt.bar(x_cl - w_cl, ari_scores, w_cl, label='Adjusted Rand Index (ARI)', color='#2b5c8f')
    plt.bar(x_cl, nmi_scores, w_cl, label='Normalized Mutual Info (NMI)', color='#7570b3')
    plt.bar(x_cl + w_cl, sil_scores, w_cl, label='Silhouette Score', color='#d95f02')
    plt.xticks(x_cl, models_cl, fontsize=14)
    plt.ylabel('Validation Score', fontsize=15, fontweight='bold')
    plt.title('Clustering Quality Comparison Across Algorithms', fontsize=16, fontweight='bold')
    plt.legend(loc='upper right', fontsize=13)
    plt.grid(axis='y', alpha=0.3)
    plt.tight_layout()
    plt.savefig(resolve_out('Ex8/Clustering_Comparison.eps'), format='eps', dpi=600)
    plt.close()

    print('Saved all Ex8 figures at 600 DPI vector EPS.')
    df_comp = pd.DataFrame({
        'Algorithm': models_cl,
        'Silhouette': sil_scores,
        'ARI': ari_scores,
        'NMI': nmi_scores
    })
    print(df_comp.to_string(index=False))
    return df_comp

In [3]:
# Master Execution Cell
ex8_results = run_experiment_8()
print('=== EXPERIMENT 8 EXECUTION COMPLETE ===')

=== LAUNCHING EXPERIMENT 8: HAR CLUSTERING PIPELINE ===


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


The PostScript backend does not support transparency; partially transparent artists will be rendered opaque.


Saved all Ex8 figures at 600 DPI vector EPS.
          Algorithm  Silhouette      ARI      NMI
      K-Means (k=6)    0.110554 0.415274 0.560075
             DBSCAN    0.207602 0.275431 0.422267
Hierarchical (Ward)    0.097255 0.430810 0.578246
=== EXPERIMENT 8 EXECUTION COMPLETE ===
